Load and Preprocess Data

In [29]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from datetime import datetime, timedelta

# Load the weather notebook
df = pd.read_csv('/workspaces/BlizzardX/Data/cleaned_data.csv')
df = df[df['ID']== 'USC00272302']
df['DATE'] = pd.to_datetime(df['DATE'])

# NEW: Define Cold Event as the bottom 10% of TMIN
tmin_10th_percentile = df['TMIN'].quantile(0.10)  # Find the 10th percentile value
df['Cold_Event'] = (df['TMIN'] <= tmin_10th_percentile).astype(int)  # Mark 1 if TMIN is in bottom 10%, 0 otherwise
print(f"10th percentile of TMIN: {tmin_10th_percentile:.2f}°C")  # Show the cutoff

10th percentile of TMIN: -10.00°C


Feature Engineering

In [30]:


df['Lag1_TMIN'] = df['TMIN'].shift(1)
df['Lag1_TMAX'] = df['TMAX'].shift(1)
df['Rolling3_TMIN'] = df['TMIN'].rolling(window=3).mean()
df['Day_of_Year'] = df['DATE'].dt.dayofyear
# Add new features
df['SNOW'] = df['SNOW']  # Daily snowfall
df['SNWD'] = df['SNWD']  # Snow depth
df['ELEVATION'] = df['ELEVATION']  # Elevation of the station
df['LATITUDE'] = df['LATITUDE']  # Latitude of the station
df['LONGITUDE'] = df['LONGITUDE']  # Longitude of the station (assuming this is what you meant by coordinates)
df = df.dropna()  # Remove rows with NaN values introduced by shifting/rolling

Train-Test Split

In [31]:


train = df[df['DATE'] <= '2019-12-31']
test = df[df['DATE'] >= '2020-01-01']
X_train = train[['Lag1_TMIN', 'Lag1_TMAX', 'Rolling3_TMIN', 'Day_of_Year', 'SNOW', 'SNWD', 'ELEVATION', 'LATITUDE', 'LONGITUDE']]
y_train = train['Cold_Event']
X_test = test[['Lag1_TMIN', 'Lag1_TMAX', 'Rolling3_TMIN', 'Day_of_Year', 'SNOW', 'SNWD', 'ELEVATION', 'LATITUDE', 'LONGITUDE']]
y_test = test['Cold_Event']

Scale Features

In [32]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Define and Evaluate Multiple Models with Split

In [33]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=42, use_label_encoder=False, eval_metric='logloss'),
    'SVM': SVC(probability=True, random_state=42)
}

tscv = TimeSeriesSplit(n_splits=5)
results = {}

print("Evaluating models with train data up to 2019-12-31 and test data from 2020-01-01 onward:")
for name, model in models.items():
    cv_scores = []
    for train_idx, val_idx in tscv.split(X_train_scaled):
        X_t, X_v = X_train_scaled[train_idx], X_train_scaled[val_idx]
        y_t, y_v = y_train.iloc[train_idx], y_train.iloc[val_idx]
        model.fit(X_t, y_t)
        y_pred = model.predict(X_v)
        cv_scores.append(accuracy_score(y_v, y_pred))
    cv_accuracy = np.mean(cv_scores)

    model.fit(X_train_scaled, y_train)
    y_pred_test = model.predict(X_test_scaled)
    test_accuracy = accuracy_score(y_test, y_pred_test)

    results[name] = {'CV Accuracy': cv_accuracy, 'Test Accuracy': test_accuracy}
    print(f"{name} - CV Accuracy (train up to 2019): {cv_accuracy:.2f}, Test Accuracy (2020 onward): {test_accuracy:.2f}")

Evaluating models with train data up to 2019-12-31 and test data from 2020-01-01 onward:
Logistic Regression - CV Accuracy (train up to 2019): 0.95, Test Accuracy (2020 onward): 0.96
Random Forest - CV Accuracy (train up to 2019): 0.94, Test Accuracy (2020 onward): 0.96


/usr/local/python/3.11.11/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [17:44:15] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/python/3.11.11/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [17:44:15] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/python/3.11.11/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [17:44:16] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/python/3.11.11/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [17:44:16] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/python/3.11.11/lib/python3.11

XGBoost - CV Accuracy (train up to 2019): 0.94, Test Accuracy (2020 onward): 0.95
SVM - CV Accuracy (train up to 2019): 0.94, Test Accuracy (2020 onward): 0.95


Identify Best Model and Predict for Next 7 Days Dynamically

In [34]:
best_model_name = max(results, key=lambda x: results[x]['Test Accuracy'])
best_model = models[best_model_name]
print(f"\nBest Model: {best_model_name}")
print(f"CV Accuracy: {results[best_model_name]['CV Accuracy']:.2f}, Test Accuracy: {results[best_model_name]['Test Accuracy']:.2f}")

# Update X_full with new features
X_full = df[['Lag1_TMIN', 'Lag1_TMAX', 'Rolling3_TMIN', 'Day_of_Year', 'SNOW', 'SNWD', 'ELEVATION', 'LATITUDE', 'LONGITUDE']]
y_full = df['Cold_Event']
X_full_scaled = scaler.fit_transform(X_full)
best_model.fit(X_full_scaled, y_full)

current_date = datetime(2025, 3, 19)
last_data_date = df['DATE'].max()  # e.g., 2025-03-03 in your sample
start_date = max(last_data_date, current_date - timedelta(days=1)) + timedelta(days=1)
future_dates = pd.date_range(start=start_date, periods=7, freq='D')
future_data = []

last_day = df[df['DATE'] == last_data_date].iloc[0]
for date in future_dates:
    features = {
        'Lag1_TMIN': last_day['TMIN'],  # Static: last day's TMIN
        'Lag1_TMAX': last_day['TMAX'],  # Static: last day's TMAX
        'Rolling3_TMIN': last_day['Rolling3_TMIN'],  # Static: last 3-day mean
        'Day_of_Year': date.dayofyear,  # Dynamic: updates daily
        'SNOW': last_day['SNOW'],  # Static: last day's snowfall
        'SNWD': last_day['SNWD'],  # Static: last day's snow depth
        'ELEVATION': last_day['ELEVATION'],  # Constant: station-specific
        'LATITUDE': last_day['LATITUDE'],   # Constant: station-specific
        'LONGITUDE': last_day['LONGITUDE']  # Constant: station-specific
    }
    future_data.append(features)
    # Static updates (no forecast logic for simplicity)
    last_day['TMIN'] = last_day['TMIN']
    last_day['TMAX'] = last_day['TMAX']

future_df = pd.DataFrame(future_data)
future_scaled = scaler.transform(future_df)

cold_probabilities = best_model.predict_proba(future_scaled)[:, 1]
predictions = best_model.predict(future_scaled)


Best Model: Logistic Regression
CV Accuracy: 0.95, Test Accuracy: 0.96


In [35]:
print(f"\nDaily Cold Event Predictions for the Next 7 Days starting {start_date.strftime('%Y-%m-%d')} (using {best_model_name}):")
for date, prob, pred in zip(future_dates, cold_probabilities, predictions):
    print(f"{date.strftime('%Y-%m-%d')}: Probability = {prob*100:.2f}%, Cold Event = {'Yes' if pred == 1 else 'No'}")


Daily Cold Event Predictions for the Next 7 Days starting 2025-04-01 (using Logistic Regression):
2025-04-01: Probability = 0.78%, Cold Event = No
2025-04-02: Probability = 0.78%, Cold Event = No
2025-04-03: Probability = 0.77%, Cold Event = No
2025-04-04: Probability = 0.77%, Cold Event = No
2025-04-05: Probability = 0.77%, Cold Event = No
2025-04-06: Probability = 0.77%, Cold Event = No
2025-04-07: Probability = 0.77%, Cold Event = No
